In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Load the Dataset

In [37]:
df = pd.read_csv("data/taxi_pickups_area.csv")
df.shape

(8064, 79)

Convert Trip Start Timestamp to datetime object

In [3]:
areas = [c for c in df.columns.to_list() if c != 'Trip Start Timestamp']
df['Trip Start Timestamp'] = pd.to_datetime(df['Trip Start Timestamp'])

Train validation split

In [38]:
train_size = int(len(df) * 0.8)

train = df[:train_size]
val = df[train_size:]

train.shape
print(train["Pickup Community Area_25"].hasnans)

False


In [5]:
val.shape

(1613, 79)

Time Series Preprocessing

Check Missing values and time frequency

In [5]:
# missing = train[train.isnan()].count()
train[train.isna()].count()

Trip Start Timestamp        0
Pickup Community Area_0     0
Pickup Community Area_1     0
Pickup Community Area_2     0
Pickup Community Area_3     0
                           ..
Pickup Community Area_73    0
Pickup Community Area_74    0
Pickup Community Area_75    0
Pickup Community Area_76    0
Pickup Community Area_77    0
Length: 79, dtype: int64

no missing values detected

Check time Frequency

In [6]:
pd.infer_freq(train[train.columns[0]])

'15min'

Time Frequency constant 15-min interval

Handle anomalies with STL Decomposition for each area

In [39]:
from statsmodels.tsa.seasonal import STL

# def detect_anomalies(serie, robust=True, period=672, threshold=3.0): # an be used!

#     stl = STL(serie, robust=robust, period=period)
#     result = stl.fit()
#     resid = result.resid

#     # Robust center/scale — mean/std get skewed by the very anomalies we want to find
#     median = resid.median()
#     mad = np.median(np.abs(resid - median))
#     mad_std = mad * 1.4826 if mad != 0 else resid.std()  # fallback if MAD is 0

#     lower = median - (threshold * mad_std)
#     upper = median + (threshold * mad_std)

#     return serie[(resid < lower) | (resid > upper)]


def detect_anomalies(serie, robust=True, period=672):

    stl = STL(serie, robust=robust, period=period)

    result = stl.fit()

    resid =  result.resid

    resid_m = resid.mean()
    resid_dv = resid.std()

    lower = resid_m - (3 * resid_dv) 
    upper = resid_m + (3 * resid_dv)

    return serie[(resid < lower) | (resid > upper)]


for area in areas:
    
    serie = train[area].copy()

    anomalies = detect_anomalies(serie)

    anomalies
    
    serie.loc[anomalies.index] = np.nan

    serie = serie.interpolate(method="linear")

    train[area] = serie

print(train["Pickup Community Area_25"].hasnans)

KeyboardInterrupt: 

save cleaned train time series

In [6]:
import joblib
joblib.dump(train, "cleaned_train.pkl")

['cleaned_train.pkl']

Load Cleaned train set

In [40]:
import joblib
train = joblib.load("cleaned_train.pkl")
# print(train["Pickup Community Area_25"].hasnans)

True


In [ ]:
Log Transformation

In [36]:
print(train["Pickup Community Area_25"].hasnans)

# transformed_train = train.copy()

# for area in areas:
#     transformed_train[area] = np.log1p(transformed_train[area] + 1)

True


Check and make time series stationary for each Area with Augmented Dickey fuller test and differencing

In [7]:
from statsmodels.tsa.stattools import adfuller
# import matplotlib.pyplot as plt


def check_stationarity(serie, significance_level=0.05):
    """
    Checks if a time series is stationary using the ADF test.
    Handles NaNs automatically before testing.
    """
    # Drop missing values caused by differencing
    clean_serie = pd.Series(serie).dropna()
    
    # Handle edge case: empty series or zero variance
    if len(clean_serie) < 10 or clean_serie.nunique() <= 1:
        return False

    result = adfuller(clean_serie, autolag='AIC')
    p_value = result[1]

    return p_value < significance_level


def make_stationary_diff(serie, max_diff=3, significance_level=0.05):
    """
    Iteratively differences a time series until it becomes stationary 
    or reaches max_diff. Returns the transformed series and total differences applied.
    """
    current_serie = pd.Series(serie).copy()
    diff_count = 0

    while diff_count < max_diff:
        if check_stationarity(current_serie, significance_level):
            break
            
        current_serie = current_serie.diff().dropna()
        diff_count += 1

    return current_serie, diff_count

try:
    for area in areas:
        serie, diff_count = make_stationary_diff(transformed_train[area])
        transformed_train[area] = serie
except Exception as e:
    print(f"error : {str(e)}")


Fit Naive averages using Global average for each Area

In [8]:
global_averages = transformed_train[areas].mean()
global_averages

Pickup Community Area_0     17.910944
Pickup Community Area_1      1.245311
Pickup Community Area_2      1.219036
Pickup Community Area_3      3.026818
Pickup Community Area_4      0.969152
                              ...    
Pickup Community Area_73     0.102000
Pickup Community Area_74     0.000000
Pickup Community Area_75     0.044567
Pickup Community Area_76    35.838862
Pickup Community Area_77     2.404278
Length: 78, dtype: float64

Forecast each Area

In [10]:
forecast_matrix = pd.Series(index=areas)
for area in areas:
    forecast_matrix[area] = np.expm1(global_averages[area]) - 1

forecast_matrix

Pickup Community Area_0     6.006537e+07
Pickup Community Area_1     1.474014e+00
Pickup Community Area_2     1.383923e+00
Pickup Community Area_3     1.863147e+01
Pickup Community Area_4     6.357086e-01
                                ...     
Pickup Community Area_73   -8.926169e-01
Pickup Community Area_74   -1.000000e+00
Pickup Community Area_75   -9.544253e-01
Pickup Community Area_76    3.669611e+15
Pickup Community Area_77    9.070439e+00
Length: 78, dtype: float64

Compute Mean Absolute Error using validation

In [11]:
from sklearn.metrics import mean_absolute_error

all_mea = []
for area in areas:
    area_pred = np.ones(len(val)) * forecast_matrix[area]
    mae = mean_absolute_error(area_pred, val[area])
    all_mea.append(mae)

average_mae = np.array(all_mea).mean()
print(f"average of all area mean absolute error : {average_mae}")

average of all area mean absolute error : 1.6164661614024093e+41


Load Taxi submission Dataset

In [12]:
sub_df = pd.read_csv("data/taxi_submission_file.csv")

Forecast with naive averages and save

In [13]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * forecast_matrix[area]
    sub_pred[area] = np.floor(area_pred)


sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
1,2019-06-24 00:15:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
2,2019-06-24 00:30:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
3,2019-06-24 00:45:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
4,2019-06-24 01:00:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
668,2019-06-30 23:00:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
669,2019-06-30 23:15:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0
670,2019-06-30 23:30:00,60065368.0,1.0,1.0,18.0,0.0,-1.0,69998.0,594.0,1.259631e+43,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,3.669611e+15,9.0


Fit Simple Moving Averages

In [25]:
transformed_train = train.copy()


window = 10
sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    avg_window = transformed_train[area].tail(window).mean() # Fit using scaled values


    inversed_avg = np.expm1(avg_window) - 1
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))

average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :0.8314631271769712


In [26]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = np.floor(area_pred)

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2019-06-24 00:15:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2019-06-24 00:30:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2019-06-24 00:45:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2019-06-24 01:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
668,2019-06-30 23:00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
669,2019-06-30 23:15:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
670,2019-06-30 23:30:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Fit Moving Averages using different window slice

In [27]:
transformed_train = train.copy()

window = 48
sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    avg_window = transformed_train[area].tail(window).mean() # Fit using scaled values


    inversed_avg = np.expm1(avg_window) - 1
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))

average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :0.6276120881403799


In [28]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = np.floor(area_pred)

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
1,2019-06-24 00:15:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
2,2019-06-24 00:30:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
3,2019-06-24 00:45:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
4,2019-06-24 01:00:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
668,2019-06-30 23:00:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
669,2019-06-30 23:15:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
670,2019-06-30 23:30:00,5.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0


Fit Weighted Moving Averages

In [29]:
transformed_train = train.copy()

window = 10
weights = np.array([i / 10 for i in range(1, window + 1)])

sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    
    avg_window = (transformed_train[area].tail(window) * weights).mean() # Fit using scaled values


    inversed_avg = np.expm1(avg_window) - 1
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))

average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :1.4062885648713077


In [30]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = np.floor(area_pred)

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
1,2019-06-24 00:15:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2,2019-06-24 00:30:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
3,2019-06-24 00:45:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
4,2019-06-24 01:00:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
668,2019-06-30 23:00:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
669,2019-06-30 23:15:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
670,2019-06-30 23:30:00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0


Simple Exponential Smoothing (SES)

In [32]:
transformed_train = train.copy()

from statsmodels.tsa.holtwinters import SimpleExpSmoothing

sma_mae = []
fit_models = []# save for submission
for area in areas:
    model = SimpleExpSmoothing(transformed_train[area], initialization_method="estimated")
    fit_model = model.fit(
    optimized=True,
    # method="L-BFGS-B",
    # maxiter=5000
)
    forecast = fit_model.forecast(steps=len(val))
    forecast = np.expm1(forecast) - 1
    # print(forecast)
    
    fit_models.append(fit_model) # save the model
    try:
       
        sma_mae.append(mean_absolute_error(forecast, val[area]))# save the area MSE
    except Exception as e:
        # sma_mae.append(mean_absolute_error(np.ones(), val[area]))
        print(area)
        print(str(e))
# average_mae = np.array(sma_mae).mean()
# print(f"average mean absolute error :{average_mae}")

/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Pickup Community Area_25
Input contains NaN.


Double Exponential Smoothing (Holt's Linear Trend)

In [42]:
from statsmodels.tsa.holtwinters import Holt

transformed_train = train.copy()

sma_mae = []
fit_models = []# save for submission
for area in areas:
    model = Holt(transformed_train[area], initialization_method="estimated")
    fit_model = model.fit(
    optimized=True,
    # method="L-BFGS-B",
    # maxiter=5000
)
    forecast = fit_model.forecast(steps=len(val))
    forecast = np.expm1(forecast) - 1
    # print(forecast)
    
    fit_models.append(fit_model) # save the model
    try:
       
        sma_mae.append(mean_absolute_error(forecast, val[area]))# save the area MSE
    except Exception as e:
        # sma_mae.append(mean_absolute_error(np.ones(), val[area]))
        print(area)
        print(str(e))

/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Pickup Community Area_25
Input contains NaN.


/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.n

In [ ]:
Triple Exponential Smoothing (Holt-Winters)

In [43]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

transformed_train = train.copy()

sma_mae = []
fit_models = []# save for submission
for area in areas:
    model = model = ExponentialSmoothing(
        transformed_train[area], 
        trend="add", 
        seasonal="add", 
        seasonal_periods=4,
        initialization_method="estimated"
)
    fit_model = model.fit(
    optimized=True,
    # method="L-BFGS-B",
    # maxiter=5000
)
    forecast = fit_model.forecast(steps=len(val))
    forecast = np.expm1(forecast) - 1
    # print(forecast)
    
    fit_models.append(fit_model) # save the model
    try:
       
        sma_mae.append(mean_absolute_error(forecast, val[area]))# save the area MSE
    except Exception as e:
        # sma_mae.append(mean_absolute_error(np.ones(), val[area]))
        print(area)
        print(str(e))

/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


Pickup Community Area_25
Input contains NaN.


/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1387: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1380: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/statsmodels/tsa/holtwinters/model.py:1387: RuntimeWarning: divide by zero encountered in log


In [ ]:
Fit and Select Best SARIMA model Based on AIC

In [ ]:
import itertools
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

p = range(0, 3)
d = range(0, 2)
q = range(0, 3)

P = range(0, 2)
D = range(0, 2)
Q = range(0, 2)

s = 672  # weekly seasonality for 15-minute data

best_aic = np.inf
best_order = None
best_seasonal_order = None
best_model = None

sarima_models = {}

for area in areas:

    best_model = None

    for order in itertools.product(p, d, q):
        for seasonal in itertools.product(P, D, Q):
    
            seasonal_order = (*seasonal, s)
    
            try:
                model = SARIMAX(
                    transformed_train[area],
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
    
                result = model.fit(disp=False)
    
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_order = order
                    best_seasonal_order = seasonal_order
                    best_model = result
    
            except Exception:
                continue
    # print("Best AIC:", best_aic)
    # print("Best order:", best_order)
    # print("Best seasonal order:", best_seasonal_order)
    sarima_model[area] = best_model

compute the average of mean absolute error of best SARIMA Model for each Area

Forecast using best SARIMA model for each Area